# Netflix Movies & TV Shows — Exploratory Data Analysis (EDA)

## Project Overview

This project explores the Netflix titles dataset to understand the composition of its movie and TV-show catalogue, content ratings, release patterns, countries, genres, and durations. It uses Python to inspect data quality, clean fields carefully, create useful features, and communicate findings through charts.

> **Dataset note:** Place `netflix_titles.csv` in the same folder as this notebook before running it. In Google Colab, upload the CSV to the session first.

### Tools
- Python
- Pandas and NumPy
- Matplotlib and Seaborn

### Important limitation
This dataset describes titles in the supplied Netflix catalogue; it does not contain viewing figures, popularity, revenue, or audience ratings. The analysis should not be interpreted as measuring what viewers watched most.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

## 2. Load the dataset

In [ ]:
DATA_PATH = Path("netflix_titles.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find netflix_titles.csv. Place the CSV beside this notebook "
        "or update DATA_PATH to the correct file location."
    )

df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns")
display(df_raw.head())

## 3. Understand the dataset

Inspect the first and last rows, column names, data types, and descriptive statistics before changing any values.

In [ ]:
display(df_raw.head())
display(df_raw.tail())
print("Column names:", df_raw.columns.tolist())
df_raw.info()
display(df_raw.describe(include="all").T)

In [ ]:
overview = pd.DataFrame({
    "data_type": df_raw.dtypes.astype(str),
    "non_null_count": df_raw.notna().sum(),
    "missing_count": df_raw.isna().sum(),
    "missing_percent": (df_raw.isna().mean() * 100).round(2),
    "unique_values": df_raw.nunique(dropna=True)
})
display(overview.sort_values("missing_percent", ascending=False))

### Initial observations
Use the output above to verify the dataset size and identify columns that need attention. Missing values are not automatically errors: for example, a title may have no director or country listed in the source.

## 4. Data quality assessment

In [ ]:
print("Exact duplicate rows:", df_raw.duplicated().sum())
missing_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_percent": (df_raw.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)
display(missing_summary)

ax = missing_summary["missing_percent"].plot(kind="bar", figsize=(11, 5))
ax.set_title("Percentage of Missing Values by Column")
ax.set_xlabel("Column")
ax.set_ylabel("Missing values (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Clean the data

Cleaning choices in this notebook:
1. Remove exact duplicate rows.
2. Trim leading/trailing spaces in text columns.
3. Convert `date_added` to a datetime value.
4. Keep missing director/cast/country values as missing in the cleaned dataset, and use a display label only when a chart requires one.
5. Keep missing `duration` and `rating` values rather than inventing values for them.

This avoids silently treating unknown information as a real director, country, or rating.

In [ ]:
df = df_raw.copy()

# Remove exact duplicate records.
df = df.drop_duplicates().copy()

# Standardize whitespace in text columns.
text_cols = df.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df[col] = df[col].str.strip()
    df[col] = df[col].replace("", np.nan)

# Parse dates safely; invalid or missing values become NaT.
if "date_added" in df.columns:
    df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")

# Ensure release_year is numeric.
if "release_year" in df.columns:
    df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")

print(f"Rows before cleaning: {len(df_raw):,}")
print(f"Rows after removing exact duplicates: {len(df):,}")
print(f"Duplicate rows remaining: {df.duplicated().sum()}")
display(df.head())

In [ ]:
cleaning_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)
display(cleaning_summary)

## 6. Feature engineering

Create year and month features from `date_added`. These describe when a title was added to the catalogue, which is different from the year it was originally released.

In [ ]:
df["added_year"] = df["date_added"].dt.year.astype("Int64")
df["added_month"] = df["date_added"].dt.month.astype("Int64")
df["added_month_name"] = df["date_added"].dt.month_name()

# Convert duration strings to numeric values for separate movie/show analysis.
df["duration_value"] = pd.to_numeric(
    df["duration"].astype("string").str.extract(r"(\d+)", expand=False),
    errors="coerce"
)

display(df[["title", "type", "date_added", "added_year", "added_month", "release_year", "duration", "duration_value"]].head(10))

## 7. Content type: Movies vs TV Shows

In [ ]:
type_counts = df["type"].value_counts(dropna=False)
type_percentages = (df["type"].value_counts(normalize=True) * 100).round(2)
display(pd.DataFrame({"titles": type_counts, "percentage": type_percentages}))

ax = sns.countplot(data=df, x="type", order=df["type"].value_counts().index, color="steelblue")
ax.set_title("Netflix Titles by Content Type")
ax.set_xlabel("Content type")
ax.set_ylabel("Number of titles")
for bar in ax.patches:
    ax.annotate(f"{int(bar.get_height()):,}",
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.show()

## 8. Titles added to Netflix over time

In [ ]:
added_by_year = df.dropna(subset=["added_year"]).groupby("added_year").size().sort_index()
display(added_by_year.tail(15))

ax = added_by_year.plot(kind="line", marker="o", figsize=(11, 5))
ax.set_title("Number of Titles Added to Netflix by Year")
ax.set_xlabel("Year added")
ax.set_ylabel("Number of titles")
plt.tight_layout()
plt.show()

In [ ]:
added_by_month = (df.dropna(subset=["added_month"])
                  .groupby("added_month").size()
                  .reindex(range(1, 13), fill_value=0))
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
ax = sns.barplot(x=month_labels, y=added_by_month.values, color="steelblue")
ax.set_title("Titles Added by Month (All Years Combined)")
ax.set_xlabel("Month")
ax.set_ylabel("Number of titles")
plt.tight_layout()
plt.show()

**Interpretation tip:** Counts by `date_added` show additions recorded in this dataset, not necessarily every title Netflix has ever offered. Missing dates are excluded from these time-based charts.

## 9. Release year analysis

In [ ]:
release_year_counts = df["release_year"].dropna().astype(int).value_counts().sort_index()
display(release_year_counts.tail(20))

ax = sns.histplot(data=df, x="release_year", bins=30, color="steelblue")
ax.set_title("Distribution of Titles by Release Year")
ax.set_xlabel("Release year")
ax.set_ylabel("Number of titles")
plt.tight_layout()
plt.show()

## 10. Content ratings

In [ ]:
rating_counts = df["rating"].fillna("Missing").value_counts()
display(rating_counts.to_frame("title_count"))

ax = sns.barplot(x=rating_counts.values, y=rating_counts.index, color="steelblue")
ax.set_title("Netflix Titles by Rating")
ax.set_xlabel("Number of titles")
ax.set_ylabel("Rating")
plt.tight_layout()
plt.show()

## 11. Country analysis

In [ ]:
# A title can list more than one country; explode creates one row per listed country.
country_df = df.assign(country=df["country"].str.split(", ")).explode("country")
country_df["country"] = country_df["country"].str.strip()
country_df = country_df[country_df["country"].notna() & country_df["country"].ne("")]
top_countries = country_df["country"].value_counts().head(10)
display(top_countries.to_frame("title_count"))

ax = sns.barplot(x=top_countries.values, y=top_countries.index, color="steelblue")
ax.set_title("Top 10 Countries Associated with Netflix Titles")
ax.set_xlabel("Number of title-country associations")
ax.set_ylabel("Country")
plt.tight_layout()
plt.show()

**Interpretation note:** Because a title may be associated with multiple countries, these counts are title-country associations. They should not be added together as if each title belonged to exactly one country.

## 12. Genre analysis

In [ ]:
genre_df = df.assign(genre=df["listed_in"].str.split(", ")).explode("genre")
genre_df["genre"] = genre_df["genre"].str.strip()
genre_df = genre_df[genre_df["genre"].notna() & genre_df["genre"].ne("")]
top_genres = genre_df["genre"].value_counts().head(10)
display(top_genres.to_frame("title_count"))

ax = sns.barplot(x=top_genres.values, y=top_genres.index, color="steelblue")
ax.set_title("Top 10 Genres / Categories")
ax.set_xlabel("Number of titles associated with category")
ax.set_ylabel("Genre / category")
plt.tight_layout()
plt.show()

## 13. Movie duration analysis

In [ ]:
movies = df[df["type"].eq("Movie")].copy()
movies["duration_minutes"] = pd.to_numeric(
    movies["duration"].astype("string").str.extract(r"(\d+)", expand=False),
    errors="coerce"
)
valid_movie_durations = movies.dropna(subset=["duration_minutes"])
print(f"Movies with a usable duration: {len(valid_movie_durations):,}")
display(valid_movie_durations["duration_minutes"].describe().to_frame("duration_minutes"))

ax = sns.histplot(data=valid_movie_durations, x="duration_minutes", bins=30, color="steelblue")
ax.set_title("Distribution of Movie Durations")
ax.set_xlabel("Duration (minutes)")
ax.set_ylabel("Number of movies")
plt.tight_layout()
plt.show()

## 14. TV show seasons analysis

In [ ]:
tv_shows = df[df["type"].eq("TV Show")].copy()
tv_shows["seasons"] = pd.to_numeric(
    tv_shows["duration"].astype("string").str.extract(r"(\d+)", expand=False),
    errors="coerce"
)
valid_seasons = tv_shows.dropna(subset=["seasons"])
display(valid_seasons["seasons"].describe().to_frame("seasons"))
season_counts = valid_seasons["seasons"].astype(int).value_counts().sort_index()
display(season_counts.to_frame("tv_show_count"))

ax = sns.barplot(x=season_counts.index.astype(str), y=season_counts.values, color="steelblue")
ax.set_title("TV Shows by Number of Seasons")
ax.set_xlabel("Number of seasons")
ax.set_ylabel("Number of TV shows")
plt.tight_layout()
plt.show()

## 15. Movies vs TV Shows by release year

In [ ]:
content_by_year = (
    df.dropna(subset=["release_year", "type"])
      .groupby(["release_year", "type"])
      .size()
      .reset_index(name="title_count")
)
# Focus the plot on the most recent 30 release years represented in the dataset.
max_year = int(content_by_year["release_year"].max())
plot_data = content_by_year[content_by_year["release_year"] >= max_year - 29]

ax = sns.lineplot(data=plot_data, x="release_year", y="title_count", hue="type", marker="o")
ax.set_title("Movies vs TV Shows by Release Year (Recent 30 Years in Dataset)")
ax.set_xlabel("Release year")
ax.set_ylabel("Number of titles")
plt.tight_layout()
plt.show()

## 16. Top directors

In [ ]:
# Split multi-director entries so each listed director is counted individually.
director_df = df.assign(director=df["director"].str.split(", ")).explode("director")
director_df["director"] = director_df["director"].str.strip()
director_df = director_df[director_df["director"].notna() & director_df["director"].ne("")]
top_directors = director_df["director"].value_counts().head(10)
display(top_directors.to_frame("title_count"))

ax = sns.barplot(x=top_directors.values, y=top_directors.index, color="steelblue")
ax.set_title("Top 10 Directors by Listed Titles")
ax.set_xlabel("Number of titles")
ax.set_ylabel("Director")
plt.tight_layout()
plt.show()

## 17. Key findings (calculated from the loaded data)

In [ ]:
# Generate concise findings from the actual dataset rather than hard-coding assumptions.
print(f"Unique titles/records after duplicate removal: {len(df):,}")

if not df["type"].dropna().empty:
    type_pct = df["type"].value_counts(normalize=True).mul(100)
    print("Content type distribution:")
    for content_type, count in df["type"].value_counts().items():
        print(f"  - {content_type}: {count:,} ({type_pct[content_type]:.1f}%)")

if not top_genres.empty:
    print(f"Most common listed genre/category: {top_genres.index[0]} ({top_genres.iloc[0]:,} associations)")
if not top_countries.empty:
    print(f"Country most frequently associated with titles: {top_countries.index[0]} ({top_countries.iloc[0]:,} associations)")
if not rating_counts.empty:
    print(f"Most frequent rating label (including missing as a category): {rating_counts.index[0]} ({rating_counts.iloc[0]:,})")
if not valid_movie_durations.empty:
    print(f"Median movie duration: {valid_movie_durations['duration_minutes'].median():.0f} minutes")
if not valid_seasons.empty:
    print(f"Median TV-show season count: {valid_seasons['seasons'].median():.0f}")
if not added_by_year.empty:
    print(f"Year with most recorded additions: {int(added_by_year.idxmax())} ({int(added_by_year.max()):,} titles)")

print("\nReminder: these are descriptive patterns in this dataset, not measures of audience popularity or viewing time.")

## 18. Final data-quality check

In [ ]:
final_check = pd.DataFrame({
    "rows": [len(df)],
    "columns": [df.shape[1]],
    "duplicate_rows": [int(df.duplicated().sum())],
    "missing_cells": [int(df.isna().sum().sum())],
    "missing_date_added": [int(df["date_added"].isna().sum())]
})
display(final_check)
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

## 19. Conclusion

This EDA examines Netflix catalogue records through data inspection, duplicate removal, text standardization, date parsing, feature engineering, and visual analysis. It compares content types and release patterns, and explores ratings, countries, categories, movie durations, TV-show seasons, and directors.

### Skills demonstrated
- Python data analysis with Pandas and NumPy
- Data-quality assessment and cleaning
- Feature engineering with datetime data
- Grouping, sorting, splitting, and exploding multi-value fields
- Visualization with Matplotlib and Seaborn
- Communicating descriptive findings with appropriate limitations

### Before publishing to GitHub
1. Run all cells from top to bottom with the dataset file available.
2. Confirm that every chart and output renders without errors.
3. Add a `README.md` describing the dataset source, project questions, tools, and key findings.
4. Include a few clear chart screenshots in the README if desired.
5. Do not describe this as a popularity analysis: the dataset does not contain viewing or audience metrics.